# Explanation Agent (Freddi) — Logic & Walkthrough

This notebook documents the **Explanation Agent**, the fifth handoff of the stock-move prediction pipeline. It reads the Manager's `sample_for_explanation.csv`, asks a language model (Ollama `llama3.2`) to justify each prediction, and writes a contract-faithful `explanations.csv` back to the Manager.

**Design in one line:** for each headline + predicted move, generate one plain-language sentence explaining *why the headline could justify that move* — the "explain your reasoning" step.

Key design choices, each explained in its own section below:
- **Option A** — the model explains the prediction from the headline only; it never sees the actual next-day outcome.
- **LangChain chain** (`prompt | ChatOllama | StrOutputParser`) for the LLM call — the pattern from Exercise 7 (RAG).
- **LangGraph `Agent`** wrapper (`agents/base.py`) so the Manager (Jack) can trigger it like every other agent.
- **Graceful fallback** — if the LLM is unavailable, a deterministic placeholder is produced, so a valid file is *always* handed back to Jack.

Contract: `docs/data_contracts.md` (Handoffs 4 and 5). Code: `agents/freddi_explanation.py`.

## Setup

This notebook runs **offline** (using the fallback) so it works anywhere without an Ollama server. For *real* `llama3.2` output, run it in Colab with a GPU and the dependencies installed — see the `agents/freddi_explanation_colab.ipynb` runner. The cell below makes the repo importable whether this notebook is opened from the repo root or from `docs/`.

In [ ]:
import os, sys

# Make the repo root importable + locate mock_data, from either repo root or docs/.
BASE = next(p for p in (".", "..") if os.path.isdir(os.path.join(p, "agents")))
sys.path.insert(0, os.path.abspath(BASE))
SAMPLE = os.path.join(BASE, "mock_data", "sample_for_explanation.csv")

import agents.freddi_explanation as fe
print("Loaded", fe.__name__, "\u2014 sample:", SAMPLE)

## 1. The data contract

The agent's input and output columns are fixed by `docs/data_contracts.md`. They are encoded as constants in the module — a single source of truth in code. Note that `prob_*` are read **in** (they feed the prompt) but are **not** written out, and the first five columns pass through untouched.

In [ ]:
print("INPUT  (Handoff 4):", fe.INPUT_COLUMNS)
print("OUTPUT (Handoff 5):", fe.OUTPUT_COLUMNS)
print("PASS-THROUGH      :", fe.PASSTHROUGH_COLUMNS)

## 2. Option A — explain the prediction only

The model is given the headline and the **predicted** move, and explains why the headline could justify it. It is deliberately **not** given the actual next-day outcome, because:

1. **It's realistic.** In production you generate the explanation *before* tomorrow's price exists, so the model couldn't know it.
2. **It matches the task** — "justify why the headline might predict that movement" — nothing about whether it was right.
3. **It avoids defending wrong answers.** Since the model never learns it was wrong, it can't invent excuses for a missed prediction.

The `actual_label` is still written to the output for the human graders — it just never reaches the model. The prompt is structured with the lecture's components (Persona / `### SECTIONS ###` / CAPITALS constraints / one-shot Example).

In [ ]:
print(fe.EXPLANATION_SYSTEM_PROMPT)
print("=" * 70)

# The per-row user message: note it carries the PREDICTED move, never the actual one.
example_row = {
    "article_title": "Intel delays next-gen chip launch", "predicted_label": "neutral",
    "actual_label": "down",  # present in the data, but NOT passed to the model
    "confidence": "0.53", "prob_up": "0.20", "prob_down": "0.40", "prob_neutral": "0.40",
}
msg = fe._user_message(example_row)
print("USER MESSAGE SENT TO THE MODEL:\n", msg)
print("\nactual_label ('down') leaked into the prompt?", "down" in msg.split('down:')[0].lower())

## 3. The LangChain chain and the LangGraph flow

**The LLM call** is built as a LangChain chain — the Exercise 7 pattern:

```python
ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT), ("user", "{user_input}")])
  | ChatOllama(model="llama3.2", ...)
  | StrOutputParser()
```

**The agent** is wrapped in the team's shared `Agent` base (`agents/base.py`) as a small LangGraph with three nodes:

```
START → load_sample → explain → write_output → END
```

- `load_sample` — read `sample_for_explanation.csv` into the state.
- `explain` — per row, run the LangChain chain; on any failure, fall back (next section).
- `write_output` — write `explanations.csv` with exactly the contract columns.

The Manager triggers it through one method — the same interface every agent shares:

```python
from agents.freddi_explanation import ExplanationAgent
ExplanationAgent().run(sample_for_explanation="outputs/sample_for_explanation.csv")
```

In [ ]:
print(fe.ExplanationAgent.__doc__)
print("Graph nodes:", ["load_sample", "explain", "write_output"])

## 4. The graceful fallback — "proceed to Jack even on failure"

If Ollama is unreachable (not installed, timed out, network error), the agent does **not** crash. It produces a deterministic placeholder sentence instead, so a valid `explanations.csv` is *always* written and the pipeline can always proceed to the Manager. The fallback follows Option A too — it never references the actual outcome. It is replaced automatically by real model output the moment Ollama is reachable.

In [ ]:
for pred in ("up", "down", "neutral"):
    row = {"article_title": "Example headline about a company", "predicted_label": pred}
    print(f"[{pred:7s}]", fe.fallback_explanation(row))

## 5. Run it end-to-end on mock data (offline)

The cell below drives the three nodes on the mock sample with `use_ollama=False`, so it runs here without a model and shows the exact `explanations.csv` shape the Manager receives. (In Colab with Ollama, the same code produces real `llama3.2` sentences instead of the placeholders.)

In [ ]:
import pandas as pd

state = {
    "sample_path": SAMPLE, "output_path": "explanations_demo.csv", "use_ollama": False,
    "model": fe.DEFAULT_MODEL, "base_url": fe.DEFAULT_OLLAMA_URL, "timeout": 30, "limit": None,
}
state.update(fe.load_sample(state))
state.update(fe.explain(state))
fe.write_output(state)

pd.read_csv("explanations_demo.csv")

### Getting real `llama3.2` output

To produce real explanations instead of the fallback, run on a machine with Ollama (e.g. Colab + GPU):

```bash
pip install langgraph langchain-core langchain-ollama
python agents/freddi_explanation.py --model llama3.2 \
    --input mock_data/sample_for_explanation.csv --output explanations.csv
```

or from Python:

```python
from agents.freddi_explanation import ExplanationAgent
ExplanationAgent(model="llama3.2").run(sample_for_explanation="mock_data/sample_for_explanation.csv")
```

## 6. Tests

The agent ships with `tests/test_explanation.py` — `assert`-based checks that run offline against `mock_data/`: contract-valid output columns, byte-for-byte pass-through columns, Option A (the prompt and fallback never include the actual outcome), and that a valid file is produced **even when the LLM is unavailable** (the "proceed to Jack on failure" guarantee). Run them with:

In [ ]:
!python -m pytest tests/test_explanation.py -v

## References

- `docs/data_contracts.md` — Handoff 4 (input) and Handoff 5 (output) column specs.
- `docs/architecture.md` — where the Explanation agent sits in the five-agent loop.
- `agents/freddi_explanation.py` — the agent implementation.
- `agents/base.py` — the shared LangGraph `Agent` interface.
- `agents/jack_manager.py` — the Manager that triggers this agent (same prompt-engineering + fallback pattern).
- Exercise 7 (RAG System) — the LangChain `prompt | model | parser` pattern this agent reuses.